# Tiny Text-to-Cypher Fine-Tuning with SmolLM2-360M

This notebook is the Colab-first training and evaluation workflow for a CUDA-only, QLoRA-based text-to-Cypher project.

What you will learn:
- how to structure a small but credible supervised fine-tuning pipeline
- how to prepare schema-grounded Cypher data
- how to baseline the untuned model before training
- how to run QLoRA correctly on an NVIDIA A100 in Colab
- how to evaluate with execution-based benchmarking against Neo4j


## 1. Environment Setup

Why this step matters:
- Colab runtimes are ephemeral, so the notebook must set up its own environment every time.
- Reproducibility starts with a deterministic repo clone and package install path.
- This project is intentionally CUDA-only. If CUDA is missing, the notebook should fail early.

What to inspect before continuing:
- the repo was cloned exactly once into `/content/text-to-cypher`
- the editable install succeeded
- `nvidia-smi` shows the A100 runtime you expect


In [ ]:
%pip install -q -U pip setuptools wheel

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/spabolu/text-to-cypher.git"
REPO_NAME = "text-to-cypher"
COLAB_ROOT = Path("/content")
CLONE_ROOT = COLAB_ROOT / REPO_NAME


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "cypher_slm").exists():
            return candidate
    return None


repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    if not CLONE_ROOT.exists():
        print(f"Cloning repo from {REPO_URL} to {CLONE_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(CLONE_ROOT)], check=True)
    repo_root = CLONE_ROOT

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Using repo root: {repo_root}")
%pip install -q -e {repo_root}
subprocess.run(["nvidia-smi"], check=False)


## 2. Imports, Runtime Preflight, and Notebook Configuration

Why this step matters:
- A fine-tuning notebook should verify the actual runtime before it starts loading models.
- On an A100, `bf16` is preferred for QLoRA compute.
- This is also the best place to define notebook-local configuration values that you want to edit directly in Colab.

What to inspect before continuing:
- CUDA is available
- the GPU name is correct
- the package versions look reasonable
- the Neo4j placeholders below are ready for you to fill in before the execution benchmark step


In [ ]:
import importlib.metadata as metadata
import importlib.util
import os
import time

import pandas as pd
import torch
from IPython.display import display

assert importlib.util.find_spec("cypher_slm") is not None, (
    "cypher_slm is not importable. Rerun the setup cell and make sure the repo cloned correctly."
)

from cypher_slm.config import RunConfig
from cypher_slm.data import build_training_corpus, load_public_examples, save_jsonl
from cypher_slm.evaluation import evaluate_examples, save_evaluation_records, verify_neo4j_connection
from cypher_slm.prompts import SYSTEM_PROMPT, render_user_prompt
from cypher_slm.reporting import summarize_records, write_markdown_report
from cypher_slm.synthetic import build_demo_schemas, generate_synthetic_examples
from cypher_slm.training import build_generation_pipeline, export_training_config, generate_query, train_qlora

assert torch.cuda.is_available(), "This notebook requires a CUDA runtime. In Colab, switch Runtime -> Change runtime type -> GPU."

run_config = RunConfig()
run_config.artifacts.ensure()

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
print(f"bf16 supported: {bf16_supported}")
print("Package versions:")
for package_name in ["torch", "transformers", "trl", "peft", "datasets", "bitsandbytes", "accelerate", "neo4j"]:
    try:
        print(f"  - {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"  - {package_name}: not installed")

print("\nA100-oriented training defaults:")
print(run_config.training)

print("\nThe dedicated Neo4j config cell appears later in the notebook. Populate it before running the execution-based benchmark cells.")


## 3. Build the Training Corpus

Industry-standard lesson:
- never start training before you know exactly where the data came from
- always keep a held-out test slice
- schema-held-out evaluation matters more than random row splits for this task

What to inspect here:
- how many rows came from public data vs synthetic data
- whether the held-out schema split looks plausible
- whether the saved JSONL artifact matches what you expect to train on


In [ ]:
demo_schemas = build_demo_schemas()
public_examples = load_public_examples()
synthetic_examples = generate_synthetic_examples(demo_schemas)
corpus = build_training_corpus(
    public_examples=public_examples,
    synthetic_examples=synthetic_examples,
    held_out_schema_ids=["commerce"],
    validation_fraction=0.15,
)

train_path = save_jsonl(corpus, run_config.artifacts.processed_data / "training_corpus.jsonl")
rows = [example.to_dict() for example in corpus]
df = pd.DataFrame(rows)

print(f"Saved corpus to {train_path}")
print(f"Total rows: {len(df)}")
display(df.groupby(["split", "source"]).size().reset_index(name="rows"))
display(df.groupby(["schema_id", "split"]).size().reset_index(name="rows"))


## 4. Inspect the Prompt Format

Why this matters:
- the model only learns the task you actually present to it
- prompt drift between training and evaluation will invalidate the benchmark
- structured-generation tasks benefit from a strict response contract

What to inspect here:
- the system instruction is narrow and explicit
- the user prompt contains the schema and the task request
- the assistant response is raw Cypher only


In [ ]:
sample = corpus[0]
print("SYSTEM PROMPT:\n")
print(SYSTEM_PROMPT)
print("\nUSER PROMPT:\n")
print(render_user_prompt(sample.schema_text, sample.question))
print("\nTARGET CYPHER:\n")
print(sample.cypher)


## 5. Baseline the Untuned Model Before Training

This is a critical habit.

If you do not measure the untuned model first, you cannot prove that fine-tuning helped.

What to inspect here:
- whether the base model already has partial Cypher competence
- whether failures are syntax errors, schema grounding errors, or logic errors
- whether the base generations are concise enough for a strict-output task


In [ ]:
base_generator = build_generation_pipeline(run_config.training.base_model)

baseline_preview = []
for example in corpus[:3]:
    generated = generate_query(base_generator, example.schema_text, example.question)
    baseline_preview.append(
        {
            "schema_id": example.schema_id,
            "question": example.question,
            "expected": example.cypher,
            "generated": generated,
        }
    )

display(pd.DataFrame(baseline_preview))


## 6. Configure Neo4j Execution Benchmark Inputs

Why this step exists:
- execution-based evaluation is stronger than string matching because multiple Cypher queries can be semantically equivalent
- credentials should be editable in one place when running directly in Colab
- the notebook should fail clearly if you forget to populate them

How to use this cell:
- replace the placeholder values below with your Aura connection details
- keep the placeholders in GitHub; do not commit real secrets back to the repo
- `WAIT_SECONDS` is useful when the Aura instance needs time to wake up


In [ ]:
NOTEBOOK_NEO4J_CONFIG = {
    "NEO4J_URI": "<populate-me>",
    "NEO4J_USERNAME": "<populate-me>",
    "NEO4J_PASSWORD": "<populate-me>",
    "NEO4J_DATABASE": "neo4j",
    "AURA_INSTANCEID": "<optional>",
    "AURA_INSTANCENAME": "<optional>",
    "WAIT_SECONDS": "60",
}

for key, value in NOTEBOOK_NEO4J_CONFIG.items():
    if value and not value.startswith("<"):
        os.environ[key] = value

print("Populate NOTEBOOK_NEO4J_CONFIG in this cell before running the execution-based benchmark cells.")


## 7. Baseline Execution-Based Evaluation Against Neo4j Aura

Why execution-based evaluation is better than exact string match:
- multiple Cypher queries can be semantically equivalent
- syntax-valid but logically wrong queries should not score as correct
- a realistic demo should measure model behavior against a live graph database

What to inspect here:
- connectivity succeeds after the configured wait
- syntax-valid rate and execution-success rate are separated
- result correctness is measured on the held-out `test` split only


In [ ]:
def require_populated_neo4j_env() -> dict[str, str]:
    required_keys = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]
    values = {key: os.getenv(key, "") for key in required_keys}
    missing = [key for key, value in values.items() if not value or value.startswith("<")]
    if missing:
        raise RuntimeError(
            "Populate NOTEBOOK_NEO4J_CONFIG with real values before running this cell. Missing: "
            + ", ".join(missing)
        )
    return values


neo4j_values = require_populated_neo4j_env()
WAIT_SECONDS = int(os.getenv("WAIT_SECONDS", "60"))
neo4j_database = os.getenv("NEO4J_DATABASE", "neo4j")
aura_instance_id = os.getenv("AURA_INSTANCEID", "")
aura_instance_name = os.getenv("AURA_INSTANCENAME", "")

print(f"Aura instance: {aura_instance_name or 'unknown'} ({aura_instance_id or 'unknown'})")
print(f"Neo4j URI: {neo4j_values['NEO4J_URI']}")
print(f"Waiting {WAIT_SECONDS} seconds before connecting...")
time.sleep(WAIT_SECONDS)

verify_neo4j_connection(
    uri=neo4j_values["NEO4J_URI"],
    username=neo4j_values["NEO4J_USERNAME"],
    password=neo4j_values["NEO4J_PASSWORD"],
    database=neo4j_database,
)
print("Neo4j Aura connectivity check passed.")

baseline_records = evaluate_examples(
    examples=[example for example in corpus if example.split == "test"],
    model_id=run_config.training.base_model,
    generator=lambda schema_text, question: generate_query(base_generator, schema_text, question),
    neo4j_uri=neo4j_values["NEO4J_URI"],
    neo4j_username=neo4j_values["NEO4J_USERNAME"],
    neo4j_password=neo4j_values["NEO4J_PASSWORD"],
    database=neo4j_database,
)
baseline_eval_path = save_evaluation_records(
    baseline_records,
    run_config.artifacts.reports / "baseline_eval.jsonl",
)
baseline_summary = summarize_records(baseline_records)
display(baseline_summary)
print(f"Saved baseline evaluation to {baseline_eval_path}")


## 8. Fine-Tune with QLoRA

What you are learning here:
- full fine-tuning updates every weight, which is unnecessary for a narrow specialization task
- QLoRA keeps the base model quantized and trains only small LoRA adapters
- the training config should be explicit, saved, and easy to compare across runs

What to inspect here:
- the exported config matches the run you intend to execute
- training logs show stable loss reduction
- checkpoints land in the configured artifact directory


In [ ]:
export_training_config(
    run_config.training,
    run_config.artifacts.model_outputs / "smollm2_training_config.txt",
)

trainer = train_qlora(corpus, run_config.training)
print(f"Model artifacts saved under {run_config.training.output_dir}")


## 9. Evaluate the Tuned Model

This section mirrors the baseline step.
The only honest comparison is to keep the benchmark constant and change only the model.

What to inspect here:
- the tuned model is more schema-grounded than the base model
- execution accuracy improves on the same held-out test set
- latency and error patterns remain reasonable for a deployment demo


In [ ]:
tuned_generator = build_generation_pipeline(run_config.training.output_dir)

tuned_preview = []
for example in corpus[:3]:
    generated = generate_query(tuned_generator, example.schema_text, example.question)
    tuned_preview.append(
        {
            "schema_id": example.schema_id,
            "question": example.question,
            "expected": example.cypher,
            "generated": generated,
        }
    )

display(pd.DataFrame(tuned_preview))

tuned_records = evaluate_examples(
    examples=[example for example in corpus if example.split == "test"],
    model_id=run_config.training.output_dir,
    generator=lambda schema_text, question: generate_query(tuned_generator, schema_text, question),
    neo4j_uri=neo4j_values["NEO4J_URI"],
    neo4j_username=neo4j_values["NEO4J_USERNAME"],
    neo4j_password=neo4j_values["NEO4J_PASSWORD"],
    database=neo4j_database,
)
tuned_eval_path = save_evaluation_records(
    tuned_records,
    run_config.artifacts.reports / "tuned_eval.jsonl",
)
tuned_summary = summarize_records(tuned_records)
display(tuned_summary)
print(f"Saved tuned evaluation to {tuned_eval_path}")


## 10. Export a Simple Benchmark Report

The final artifact should be understandable without opening the notebook.
That is why we export the summary table to markdown.


In [ ]:
report_path = write_markdown_report(
    tuned_summary,
    run_config.artifacts.reports / "benchmark_summary.md",
    title="SmolLM2-360M Text-to-Cypher Benchmark Summary",
)
print(f"Wrote markdown summary to {report_path}")


## Next Learning Iterations

Once this notebook works end-to-end, the next useful experiments are:
1. change the held-out schema and measure generalization shift
2. add more synthetic query families and track which families improve execution accuracy
3. compare exact-match, syntax-valid, and execution-correct metrics side by side
4. run a second training sweep with different LoRA rank or learning rate and document the tradeoff
